# ML: прогноз дебита


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine, text

import os
postgres_url = lambda: f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'admin')}:{os.getenv('POSTGRES_PASSWORD', 'admin')}@{os.getenv('POSTGRES_HOST', 'postgres')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'oil_analytics')}"

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "docker-compose.yml").exists())
FEATURES = ["avg_pressure", "avg_temperature", "avg_power_kw", "pump_runtime_hours"]


def build_dataset(engine):
    telemetry = pd.read_sql("SELECT * FROM well_telemetry", engine, parse_dates=["timestamp"])
    targets = pd.read_sql("SELECT * FROM well_targets", engine, parse_dates=["date"])
    production = pd.read_sql("SELECT * FROM production", engine, parse_dates=["date"])
    telemetry["date"] = telemetry["timestamp"].dt.normalize()
    daily = telemetry.groupby(["well_id", "date"], as_index=False).agg(
        pressure_in=("pressure_in", "mean"),
        pressure_out=("pressure_out", "mean"),
        avg_temperature=("temperature", "mean"),
        avg_power_kw=("pump_current", "mean"),
        pump_runtime_hours=("record_id", "count"),
    )
    daily["avg_pressure"] = (daily["pressure_in"] + daily["pressure_out"]) / 2
    daily = daily[["well_id", "date", *FEATURES]]
    prod = production[["well_id", "date", "pressure", "temperature", "energy_kwh", "downtime_hours"]].copy()
    prod["avg_pressure_prod"] = prod["pressure"]
    prod["avg_temperature_prod"] = prod["temperature"]
    prod["avg_power_kw_prod"] = prod["energy_kwh"] / 24
    prod["pump_runtime_hours_prod"] = 24 - prod["downtime_hours"].fillna(0)
    dataset = targets.merge(daily, on=["well_id", "date"], how="left").merge(prod, on=["well_id", "date"], how="left")
    for column in FEATURES:
        dataset[column] = dataset[column].fillna(dataset[f"{column}_prod"])
    dataset[FEATURES] = dataset[FEATURES].fillna(dataset[FEATURES].median(numeric_only=True))
    return dataset.dropna(subset=["daily_oil_ton"])


In [2]:
def run_forecast():

    engine = create_engine(postgres_url())
    dataset = build_dataset(engine)
    x_train, x_test, y_train, y_test = train_test_split(
        dataset[FEATURES],
        dataset["daily_oil_ton"],
        test_size=0.25,
        random_state=42,
    )
    models = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3),
    }
    scores = {}
    best_name = None
    best_model = None
    best_rmse = float("inf")
    for name, model in models.items():
        model.fit(x_train, y_train)
        prediction = model.predict(x_test)
        mae = mean_absolute_error(y_test, prediction)
        rmse = np.sqrt(mean_squared_error(y_test, prediction))
        scores[name] = {"mae": float(mae), "rmse": float(rmse), "r2": float(r2_score(y_test, prediction))}
        if rmse < best_rmse:
            best_name = name
            best_model = model
            best_rmse = rmse
    dataset["predicted_flow_tpd"] = best_model.predict(dataset[FEATURES])
    dataset["absolute_error"] = (dataset["daily_oil_ton"] - dataset["predicted_flow_tpd"]).abs()
    dataset["model_name"] = best_name
    output = PROJECT_ROOT / "models"
    output.mkdir(exist_ok=True)
    joblib.dump(best_model, output / "production_forecast.joblib")
    (output / "production_metrics.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
    with engine.begin() as connection:
        connection.execute(text("DROP TABLE IF EXISTS mart_ml_predictions"))
    dataset.to_sql("mart_ml_predictions", engine, if_exists="replace", index=False)
    print(json.dumps({"best_model": best_name, "metrics": scores}, indent=2))



run_forecast()


{
  "best_model": "random_forest",
  "metrics": {
    "linear_regression": {
      "mae": 2.5145191229980206,
      "rmse": 3.1139629505382858,
      "r2": 0.9243767909081383
    },
    "random_forest": {
      "mae": 0.7423656893227456,
      "rmse": 2.4630173878139945,
      "r2": 0.9526888857808629
    }
  }
}
